In [1]:
import torch 
import matplotlib.pyplot as plt 

%matplotlib inline

In [2]:
names = open("./names.txt", "r").read().splitlines()
names[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [ ]:
characters = sorted(set("".join(names)))
stoi = {c:i+1 for i, c in enumerate(characters)}
stoi["."] = 0
itos = {i:c for c, i in stoi.items()}
total_unique_chars_in_dataset = len(itos)


In [7]:
# turn text into training dataset 

BLOCK_SIZE = 3
X,Y = [],[]

for w in names:
    
    context = [0] * BLOCK_SIZE
    
    for ch in (w+"."):
        
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        
        context = context[1:]+[ix]

X = torch.tensor(X)
Y = torch.tensor(Y)
X.shape, Y.shape, X, Y

(torch.Size([228146, 3]),
 torch.Size([228146]),
 tensor([[ 0,  0,  0],
         [ 0,  0,  5],
         [ 0,  5, 13],
         ...,
         [26, 26, 25],
         [26, 25, 26],
         [25, 26, 24]]),
 tensor([ 5, 13, 13,  ..., 26, 24,  0]))

In [11]:
# Embedding Lookup Table 

embedding_dim = 2 
EMBEDDING_TABLE = torch.randn((total_unique_chars_in_dataset, embedding_dim))
EMBEDDING_TABLE[:10]

tensor([[-1.2724,  0.6163],
        [ 1.7245, -0.1723],
        [ 1.4286, -0.9928],
        [ 1.9536, -1.0461],
        [ 1.1737, -1.0337],
        [-0.8481,  0.4916],
        [-2.2918,  0.1342],
        [-0.0439,  1.4619],
        [-1.8303,  1.8552],
        [-0.7491, -0.8534]])

In [ ]:
# prepare the dataset X to their embedding value and to get the embedded valued dataset 
EMBEDDED_X = EMBEDDING_TABLE[X]
EMBEDDED_X[:10]

tensor([[[-1.2724,  0.6163],
         [-1.2724,  0.6163],
         [-1.2724,  0.6163]],

        [[-1.2724,  0.6163],
         [-1.2724,  0.6163],
         [-0.8481,  0.4916]],

        [[-1.2724,  0.6163],
         [-0.8481,  0.4916],
         [-0.8764, -2.1976]],

        [[-0.8481,  0.4916],
         [-0.8764, -2.1976],
         [-0.8764, -2.1976]],

        [[-0.8764, -2.1976],
         [-0.8764, -2.1976],
         [ 1.7245, -0.1723]],

        [[-1.2724,  0.6163],
         [-1.2724,  0.6163],
         [-1.2724,  0.6163]],

        [[-1.2724,  0.6163],
         [-1.2724,  0.6163],
         [ 0.9135,  1.2128]],

        [[-1.2724,  0.6163],
         [ 0.9135,  1.2128],
         [ 2.1389,  0.0717]],

        [[ 0.9135,  1.2128],
         [ 2.1389,  0.0717],
         [-0.7491, -0.8534]],

        [[ 2.1389,  0.0717],
         [-0.7491, -0.8534],
         [-0.8159,  0.9882]]])

In [23]:
# Shuffle and Splitting the dataset: train, dev, test :: 0.8 : 0.1 : 0.1

# Shuffling
RANDOM_SEED = 4204902
g = torch.Generator().manual_seed(RANDOM_SEED)
perm = torch.randperm(EMBEDDED_X.shape[0], generator=g)
EMBEDDED_X_SHUFFLE, Y_SHUFFLE = EMBEDDED_X[perm], Y[perm]

# Splitting

dataset_len = EMBEDDED_X_SHUFFLE.shape[0]
n1 = int(0.8 * dataset_len)
n2 = int(0.9 * dataset_len)

X_train, Y_train = EMBEDDED_X_SHUFFLE[:n1], Y_SHUFFLE[:n1]
X_dev, Y_dev = EMBEDDED_X_SHUFFLE[n1:n2], Y_SHUFFLE[n1:n2]
X_test, Y_test = EMBEDDED_X_SHUFFLE[n2:], Y_SHUFFLE[n2:]


In [24]:
print(f"{X_train.shape=}, {Y_train.shape=}\n{X_dev.shape=}, {Y_dev.shape=}\n{X_test.shape=}, {Y_test.shape=}")

X_train.shape=torch.Size([182516, 3, 2]), Y_train.shape=torch.Size([182516])
X_dev.shape=torch.Size([22815, 3, 2]), Y_dev.shape=torch.Size([22815])
X_test.shape=torch.Size([22815, 3, 2]), Y_test.shape=torch.Size([22815])


In [30]:
X_train.shape, X_train.shape[1]*X_train.shape[2]

(torch.Size([182516, 3, 2]), 6)

In [31]:
# Hidden Layer 1 
Total_Neurons_H1 = 150
W1 = torch.randn((X_train.shape[1]*X_train.shape[2], Total_Neurons_H1), generator=g)
b1 = torch.randn(W1.shape[1], generator=g)

# Hidden Layer 2
total_out_logits = total_unique_chars_in_dataset
W2 = torch.randn((W1.shape[1], total_out_logits), generator=g)
b2 = torch.randn(W2.shape[1], generator=g)


In [35]:
# parameters
parameters = [EMBEDDING_TABLE, W1, b1, W2, b2]

sum(p.nelement() for p in parameters)

5181

In [ ]:
X_train.shape, W1.shape

(torch.Size([182516, 3, 2]), torch.Size([6, 150]))

In [53]:
# checking if all shapes aligned for scalar martrix product
X_train_ = X_train.view(-1, 6)
print(f"{X_train[:3]=},\n\n{X_train_[:3]=},\n\n{X_train_.shape=}, {X_train_.dim()=}, {W1.shape=}")

X_train[:3]=tensor([[[ 1.4286, -0.9928],
         [-1.1306, -0.6506],
         [-0.7491, -0.8534]],

        [[-0.8481,  0.4916],
         [-0.1052,  0.0912],
         [ 2.1389,  0.0717]],

        [[ 0.0490, -1.4643],
         [-1.8303,  1.8552],
         [ 1.7245, -0.1723]]]),

X_train_[:3]=tensor([[ 1.4286, -0.9928, -1.1306, -0.6506, -0.7491, -0.8534],
        [-0.8481,  0.4916, -0.1052,  0.0912,  2.1389,  0.0717],
        [ 0.0490, -1.4643, -1.8303,  1.8552,  1.7245, -0.1723]]),

X_train_.shape=torch.Size([182516, 6]), X_train_.dim()=2, W1.shape=torch.Size([6, 150])


In [ ]:
# Training loops
